# Exercise 2. LoRa for Low-Resource Languages
NLP for social good is not just about reducing harmful outputs; it is also about making AI accessible across languages, not only English. Low- and medium-resource languages, from Nigerian Pidgin to Danish, are often left behind. 

```{figure} ../figures/class8/neural-space-low-resource.png
---
name: neural-space-low-resource
width: 100%
---
Fig. borrowed from [NeuralSpace blogpost](https://medium.com/neuralspace/challenges-in-using-nlp-for-low-resource-languages-and-how-neuralspace-solves-them-54a01356a71b) by Felix Laumann
```

Fine-tuning LLMs can help, but it is costly. LoRA (Low-Rank Adaptation) offers a parameter-efficient alternative, reducing trainable parameters by up to 10,000 times. In other words, rather than training all 8 billion parameters of a model like [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B-Base), LoRA updates only a small fraction.

## 2.1 Intro to LoRa?
When we're doing LoRa, we are esentially training a LoRa adapter that could technically be placed on other models (if the base architecture matches):

```{figure} ../figures/class8/lora_adapter.png
---
name: lora_adapter
width: 100%
---
From HF's [smol course](https://huggingface.co/learn/smol-course/en/unit1/3a)
```

If you're interested in the math behind this (but in an intuitive way), I encourage you to read Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/i/138081202/a-brief-introduction-to-lora). You can also read the original paper by {cite:t}`hu_lora_2021`. 

## 2.2 Setup
For the code implementation, we'll use the [PEFT](https://huggingface.co/docs/peft/en/index) and [TRL](https://huggingface.co/docs/trl/en/index) library by Hugging Face
```bash
source .venv/bin/activate
pip install peft trl
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers datasets
```

Let's import:

In [147]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

from trl import SFTTrainer, SFTConfig

## 2.3 Load Model & Data
For today's exercise, we'll try to make a smaller version of `SmolLM2` good at English to Danish machine translation

:::{admonition} You can use LoRa for much more than Translation :)
:class: dropdown, tip
As a simple introduction to LoRA, we're doing machine translation, but you can use this approach for anything you'd like really - feel free to switch out the dataset for something you'd like. Or use this notebook as a inspiration for the exam :).

See also this tutorial for instruction-tuning a danish language model using QLoRA -> [Tutorial: Finetuning Language Models](https://www.foundationmodels.dk/blog/2024/02/02/tutorial-finetuning-language-models.html)
:::

We'll load a smaller version of `SmolLM2`:

In [148]:
model_id ="HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

We'll load the Danish-English translation set, but only a subset with `[:n]` for `n` rows:

In [149]:
train_ds = load_dataset("Helsinki-NLP/opus-100", "da-en", split="train[:1000]")

Let's look at the only column, "translation" to see how it is structured: 

In [150]:
train_ds[0]

{'translation': {'da': 'På Det Blandede EØS-Udvalgs vegne',
  'en': 'For the EEA Joint Committee'}}

Let's print a few:

In [151]:
for translation in train_ds["translation"][:5]:
    print(f"EN: {translation['en']}")
    print(f"DA: {translation['da']}")
    print()

EN: For the EEA Joint Committee
DA: På Det Blandede EØS-Udvalgs vegne

EN: Metal containing by weight at least 99,9 % of lead, provided that the content by weight of any other element does not exceed the limit specified in the following table:
DA: metal, der indeholder mindst 99,9 vægtprocent bly, forudsat ingen anden bestanddel indgår i mængder, der overstiger de i nedenstående skema anførte grænseværdier:

EN: Think.
DA: Tænk.

EN: Beth...
DA: Beth...

EN: With the Human Hibernation Project, we will be able to save our best men... frozen in their prime, for use when they are needed most.
DA: Vort projekt "Menneskelig dvale" gør det muligt at holde vores bedste mænd nedfrosset i deres bedste tilstand, for at bruge dem efter behov.



## 1.3 Prompt Templating
We want a `prompt` column that inserts the English sentence as the `Source` and the Danish example as the `target`in this formatting by{cite:t}`alves_steering_2023`:
```{figure} ../figures/class8/prompt-template-alves.png
---
name: prompt-template-alves
width: 80%
---
Prompt template by {cite:t}`alves_steering_2023`
```
X should be "English" and Y should be "Danish" in our context.

### Your Turn: Formatting the Prompt
:::{admonition} HANDS-ON
:class: red
1. Create a function called `def format_prompt(example)`
    - It should process a single row `example` in our dataset
    - Use the prompt template above to create a `prompt` with English as the source & Danish as the target.
    - Return a dictionary entry `{"prompt": prompt}`

2. Test the function on a single example in `train_ds`, printing the prompt and completion"
:::

#### Solution

In [152]:
# define function
def preprocess_function(example):
    translation = example["translation"]
    return {
        "prompt": [{"role": "user", "content": f"Translate to Danish: {translation['en']}"}],
        "completion": [
            {"role": "assistant", "content": f"{translation['da']}"}
        ],  
    }

In [153]:
def preprocess_function(example):
    translation = example["translation"]
    return {"messages": [{"role": "user", "content": f"Translate to Danish: {translation['en']}"}, {"role": "assistant", "content": f"{translation['da']}"}]}

### Adding a Prompt Column 
We can now add the prompt column to our ds using our new `format_prompt` column:

In [154]:
formatted_train_ds = train_ds.map(preprocess_function, batched=False)

In [155]:
formatted_train_ds

Dataset({
    features: ['translation', 'messages'],
    num_rows: 1000
})

In [156]:
formatted_train_ds = formatted_train_ds.remove_columns(["translation"])

## 1.4 LoRa Config & Training
We'll start by configuring LoRA: 

In [157]:
rank = 16
peft_config = LoraConfig(
    r=rank,
    lora_alpha=rank * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [158]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="lora-adapter",
        num_train_epochs=1,
        per_device_train_batch_size=2,
    ),
    train_dataset=formatted_train_ds,
    peft_config=peft_config
)
trainer.train()

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,4.165300
20,4.083000
30,3.974100
40,3.976400
50,3.888400
60,3.838500
70,3.903500
80,3.902300
90,3.441200
100,3.497800


TrainOutput(global_step=500, training_loss=2.7605238037109374, metrics={'train_runtime': 76.1311, 'train_samples_per_second': 13.135, 'train_steps_per_second': 6.568, 'total_flos': 40066484944896.0, 'train_loss': 2.7605238037109374, 'epoch': 1.0})

In [163]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "HuggingFaceTB/SmolLM2-135M-Instruct"
adapter_path = "lora-adapter/checkpoint-500"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(base_model)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

messages = [
    {"role": "user", "content": "Translate to Danish: I love natural language processing."}
]

# flatten messages into a string
prompt = ""
for msg in messages:
    prompt += f"[{msg['role'].upper()}]: {msg['content']}\n"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[USER]: Translate to Danish: I love natural language processing.

**Translation Style:**
- **Formal**: "I love natural language processing." (Formal, objective, objective language)
- **Informal**: "I love natural language processing." (Informal, conversational, informal language)

